In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# File paths
base_path = "C:/Users/Admin/Desktop/Mac files/Signal Processing/CMaps"
columns = ["engine_id", "cycle"] + \
          [f"setting_{i}" for i in range(1, 4)] + \
          [f"sensor_{i}" for i in range(1, 22)]

# Feature Engineering
def feature_engineering(df):
    sensors_to_use = [f'sensor_{i}' for i in range(2, 22) if f'sensor_{i}' not in ['sensor_5', 'sensor_6', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']]
    df['cycle_to_failure'] = df.groupby('engine_id')['cycle'].transform(lambda x: x.max() - x) / df['cycle']
    for sensor in sensors_to_use:
        df[f'{sensor}_ma'] = df.groupby('engine_id')[sensor].transform(lambda x: x.rolling(window=5, min_periods=1).mean())
    df.fillna(df.mean(), inplace=True)
    return df

# Data Preprocessing
def preprocess_data(train_df, test_df, rul_df, n_components=15):
    drop_cols = ['setting_3', 'sensor_1', 'sensor_5', 'sensor_6', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
    train_df = train_df.drop(columns=drop_cols)
    test_df = test_df.drop(columns=drop_cols)
    train_df = feature_engineering(train_df)
    test_df = feature_engineering(test_df)
    feature_cols = [col for col in train_df.columns if col not in ['engine_id', 'cycle', 'RUL']]
    scaler = StandardScaler()
    train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
    test_df[feature_cols] = scaler.transform(test_df[feature_cols])
    pca = PCA(n_components=n_components)
    train_pca = pca.fit_transform(train_df[feature_cols])
    test_pca = pca.transform(test_df[feature_cols])
    train_pca_df = pd.DataFrame(train_pca, columns=[f'pca_{i+1}' for i in range(n_components)])
    test_pca_df = pd.DataFrame(test_pca, columns=[f'pca_{i+1}' for i in range(n_components)])
    train_df = pd.concat([train_df[['engine_id', 'cycle']].reset_index(drop=True), train_pca_df], axis=1)
    test_df = pd.concat([test_df[['engine_id', 'cycle']].reset_index(drop=True), test_pca_df], axis=1)
    train_df['RUL'] = train_df.groupby('engine_id')['cycle'].transform(lambda x: x.max() - x).clip(upper=125)
    test_rul = []
    for engine_id in test_df['engine_id'].unique():
        max_cycle = test_df[test_df['engine_id'] == engine_id]['cycle'].max()
        initial_rul = rul_df.loc[engine_id - 1, 'RUL']
        test_rul.extend([initial_rul + max_cycle - cycle for cycle in test_df[test_df['engine_id'] == engine_id]['cycle']])
    test_df['RUL'] = test_rul
    test_df['RUL'] = test_df['RUL'].clip(upper=125)
    return train_df, test_df

# Create graph data for PyTorch Geometric
def create_graph_data(df):
    data_list = []
    for i, row in df.iterrows():
        x = torch.tensor(row.iloc[2:-1].values, dtype=torch.float32).unsqueeze(0)
        y = torch.tensor(row.iloc[-1], dtype=torch.float32).unsqueeze(0)
        edge_index = torch.tensor(
            [[i, i] for i in range(x.size(0))], dtype=torch.long
        ).t()  # Self-loops for each node
        data = Data(x=x, edge_index=edge_index, y=y)
        data_list.append(data)
    return data_list

# Define GNN Model
class GCNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GCNModel, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
    
    def forward(self, x, edge_index, batch):
        x = self.relu(self.conv1(x, edge_index))
        x = self.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        x = self.fc(x)
        return x

# Calculate accuracy within ±20 cycles
def calculate_margin_accuracy(predictions, actuals, margin=20):
    correct = sum(abs(pred - actual) <= margin for pred, actual in zip(predictions, actuals))
    return correct / len(actuals) * 100

# Training and Testing Functions
def train_model(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0
    for data in dataloader:
        data = data.to(device)
        optimizer.zero_grad()
        predictions = model(data.x, data.edge_index, data.batch)
        loss = criterion(predictions.view(-1), data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def test_model(model, dataloader):
    model.eval()
    predictions, actuals = [], []
    with torch.no_grad():
        for data in dataloader:
            data = data.to(device)
            pred = model(data.x, data.edge_index, data.batch)
            predictions.extend(pred.view(-1).tolist())
            actuals.extend(data.y.tolist())
    return predictions, actuals

# Hyperparameters
input_dim = 15
hidden_dim = 512
output_dim = 1

# List of dataset identifiers
dataset_ids = ["FD001", "FD002", "FD003", "FD004"]

# Results storage
results = {}

# Loop through each dataset
for FD_ID in dataset_ids:
    print(f"Processing Dataset: {FD_ID}")
    
    train_file = f"{base_path}/train_{FD_ID}.txt"
    test_file = f"{base_path}/test_{FD_ID}.txt"
    rul_file = f"{base_path}/RUL_{FD_ID}.txt"
    
    # Load and preprocess data
    train_df = pd.read_csv(train_file, sep='\s+', header=None, names=columns)
    test_df = pd.read_csv(test_file, sep='\s+', header=None, names=columns)
    rul_df = pd.read_csv(rul_file, sep='\s+', header=None, names=["RUL"])
    train_processed, test_processed = preprocess_data(train_df, test_df, rul_df)
    
    # Create graph datasets using full data
    train_graphs = create_graph_data(train_processed)  # Full training data
    test_graphs = create_graph_data(test_processed)    # Full testing data
    
    # Create data loaders
    train_loader = GeoDataLoader(train_graphs, batch_size=32, shuffle=True)
    test_loader = GeoDataLoader(test_graphs, batch_size=32, shuffle=False)
    
    # Initialize and train the model
    model = GCNModel(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss().to(device)
    
    for epoch in range(30):  # Reduced epochs for testing
        train_loss = train_model(model, train_loader, optimizer, criterion)
        val_predictions, val_actuals = test_model(model, test_loader)
        val_loss = np.sqrt(mean_squared_error(val_actuals, val_predictions))
        print(f"Dataset {FD_ID} - Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Validation RMSE: {val_loss:.4f}")
    
    # Store results for analysis
    results[FD_ID] = {"predictions": val_predictions, "actuals": val_actuals}

# Analyze and print results
for FD_ID, data in results.items():
    predictions, actuals = data["predictions"], data["actuals"]
    margin_accuracy = calculate_margin_accuracy(predictions, actuals, margin=20)
    print(f"\nResults for Dataset {FD_ID}:")
    print(f"  Final RMSE: {np.sqrt(mean_squared_error(actuals, predictions)):.2f}")
    print(f"  Accuracy within ±20 cycles: {margin_accuracy:.2f}%")


Using device: cuda
Processing Dataset: FD001
Dataset FD001 - Epoch 1, Train Loss: 907.8071, Validation RMSE: 18.1117
Dataset FD001 - Epoch 2, Train Loss: 312.6924, Validation RMSE: 17.1687
Dataset FD001 - Epoch 3, Train Loss: 267.2041, Validation RMSE: 17.8855
Dataset FD001 - Epoch 4, Train Loss: 237.3650, Validation RMSE: 17.1596
Dataset FD001 - Epoch 5, Train Loss: 217.2042, Validation RMSE: 17.7587
Dataset FD001 - Epoch 6, Train Loss: 199.4188, Validation RMSE: 19.0286
Dataset FD001 - Epoch 7, Train Loss: 184.8589, Validation RMSE: 19.6221
Dataset FD001 - Epoch 8, Train Loss: 170.0069, Validation RMSE: 19.2001
Dataset FD001 - Epoch 9, Train Loss: 157.4765, Validation RMSE: 21.2397
Dataset FD001 - Epoch 10, Train Loss: 153.1904, Validation RMSE: 22.6937
Dataset FD001 - Epoch 11, Train Loss: 145.7537, Validation RMSE: 26.2416
Dataset FD001 - Epoch 12, Train Loss: 141.0916, Validation RMSE: 26.9962
Dataset FD001 - Epoch 13, Train Loss: 140.3097, Validation RMSE: 27.5220
Dataset FD001 -

In [2]:
# Function to calculate accuracy for RUL values under 20
def calculate_accuracy_under_20(predictions, actuals, margin=10):
    filtered_predictions = []
    filtered_actuals = []
    
    # Filter only those predictions and actuals where RUL < 20
    for pred, actual in zip(predictions, actuals):
        if actual < 20:
            filtered_predictions.append(pred)
            filtered_actuals.append(actual)
    
    # Calculate accuracy within the margin for filtered data
    correct = sum(abs(pred - actual) <= margin for pred, actual in zip(filtered_predictions, filtered_actuals))
    accuracy = (correct / len(filtered_actuals)) * 100 if filtered_actuals else 0
    return accuracy

# Analyze and print results specifically for RUL < 20
for FD_ID, data in results.items():
    predictions, actuals = data["predictions"], data["actuals"]
    margin_accuracy = calculate_margin_accuracy(predictions, actuals, margin=10)
    accuracy_under_20 = calculate_accuracy_under_20(predictions, actuals, margin=10)
    
    print(f"\nResults for Dataset {FD_ID}:")
    print(f"  Final RMSE: {np.sqrt(mean_squared_error(actuals, predictions)):.2f}")
    print(f"  Accuracy within ±10 cycles: {margin_accuracy:.2f}%")
    print(f"  Accuracy for RUL < 20 within ±10 cycles: {accuracy_under_20:.2f}%")


Results for Dataset FD001:
  Final RMSE: 26.81
  Accuracy within ±10 cycles: 55.70%
  Accuracy for RUL < 20 within ±10 cycles: 75.70%

Results for Dataset FD002:
  Final RMSE: 32.68
  Accuracy within ±10 cycles: 50.15%
  Accuracy for RUL < 20 within ±10 cycles: 88.69%

Results for Dataset FD003:
  Final RMSE: 21.67
  Accuracy within ±10 cycles: 59.67%
  Accuracy for RUL < 20 within ±10 cycles: 84.26%

Results for Dataset FD004:
  Final RMSE: 27.84
  Accuracy within ±10 cycles: 56.67%
  Accuracy for RUL < 20 within ±10 cycles: 87.72%
